# LLM Fine-Tuning Benchmark Lab

In [ ]:
# Update Repo
REPO_URL = "https://github.com/ronitnpatel/LLM-Fine-Tune-Benchmark.git"

import os, sys, subprocess, pathlib
from google.colab import drive

drive.mount('/content/drive')
repo_dir = pathlib.Path('/content/llm-finetuning-benchmark')
required_source = pathlib.Path('src/models/model_loader.py')
if repo_dir.exists() and not (repo_dir / required_source).is_file():
    print('Existing checkout is missing src/models; using a fresh checkout directory.')
    repo_dir = pathlib.Path('/content/llm-finetuning-benchmark-checkout')
if not repo_dir.exists():
    if "OWNER/REPOSITORY" in REPO_URL:
        raise ValueError("Set REPO_URL to a GitHub repository containing src/, configs/, and requirements.txt.")
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
if not (repo_dir / required_source).is_file():
    raise FileNotFoundError(f"Project source not found at {repo_dir / required_source}. Push or upload the complete project, then rerun setup.")
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
# Persist the Hugging Face cache on Drive so model/data downloads survive runtime resets.
os.environ["HF_HOME"] = "/content/drive/MyDrive/llm-finetuning-benchmark/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository:", repo_dir)
print("Drive-backed Hugging Face cache:", os.environ["HF_HOME"])

Mounted at /content/drive
Repository: /content/llm-finetuning-benchmark
Drive-backed Hugging Face cache: /content/drive/MyDrive/llm-finetuning-benchmark/hf-cache


## Runtime and configuration

In [3]:
import gc, json, logging, random, traceback, hashlib, platform
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import yaml
import torch
from huggingface_hub import HfApi

from src.models.model_loader import choose_model, load_model, load_tokenizer, model_metadata, preflight_full_finetuning
from src.data.dataset_loader import load_splits
from src.data.preprocessing import tokenize_example
from src.training.lora_trainer import add_lora
from src.training.qlora_trainer import add_qlora
from src.training.full_finetuning import prepare_full_finetuning
from src.training.trainer import train_experiment, count_trainable_parameters
from src.evaluation.evaluator import evaluate_generation, evaluate_causal_loss, generate_generalization_samples
from src.benchmarking.benchmark import save_results, generate_plots
from scripts.generate_report import build_report

logging.basicConfig(level=logging.INFO)
with open('configs/model_config.yaml', encoding='utf-8') as f: model_cfg = yaml.safe_load(f)
with open('configs/dataset_config.yaml', encoding='utf-8') as f: data_cfg = yaml.safe_load(f)
with open('configs/training_config.yaml', encoding='utf-8') as f: train_cfg = yaml.safe_load(f)
if RUN_MODE == 'smoke':
    data_cfg['max_examples'] = min(data_cfg['max_examples'], train_cfg['smoke_examples'] + 12)
    train_cfg['epochs'] = train_cfg['smoke_epochs']
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Choose a Colab GPU runtime; training is not performed on the local machine.')
random.seed(train_cfg['seed']); torch.manual_seed(train_cfg['seed']); torch.cuda.manual_seed_all(train_cfg['seed'])
selected_model, hardware = choose_model(model_cfg['primary_model'], model_cfg['fallback_model'], model_cfg['full_finetune_min_free_gib'])
preflight = preflight_full_finetuning(selected_model, model_cfg['revision'], min(data_cfg['max_length'], 512))
preflight_history = {selected_model: preflight}
if not preflight['fits'] and model_cfg['fallback_model'] and model_cfg['fallback_model'] != selected_model:
    print('Primary model failed full-tuning preflight; trying the configured smaller fallback before any benchmark run.')
    primary_failure = preflight
    selected_model = model_cfg['fallback_model']
    preflight = preflight_full_finetuning(selected_model, model_cfg['revision'], min(data_cfg['max_length'], 512))
    preflight_history[selected_model] = preflight
    preflight_history[model_cfg['primary_model']] = primary_failure
    hardware['selection_reason'] = 'primary model failed scratch full-finetuning probe; smaller fallback selected before baseline'
hardware['full_finetuning_preflight'] = preflight_history
if not preflight['fits']:
    diagnostic_dir = pathlib.Path('/content/drive/MyDrive/llm-finetuning-benchmark/results/reports'); diagnostic_dir.mkdir(parents=True, exist_ok=True)
    (diagnostic_dir / 'full_finetuning_preflight_failure.json').write_text(json.dumps({'primary': model_cfg['primary_model'], 'fallback': model_cfg['fallback_model'], 'hardware': hardware}, indent=2), encoding='utf-8')
    raise RuntimeError(f'Full fine-tuning preflight failed for the configured model; stopped before baseline. Diagnostic saved to storage. Hardware: {hardware}')
print(json.dumps(hardware, indent=2))
print('Selected model for every experiment:', selected_model)

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/llm-finetuning-benchmark')
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
RESULTS_ROOT = DRIVE_ROOT / 'results'
METRICS_DIR = RESULTS_ROOT / 'metrics'; PLOTS_DIR = RESULTS_ROOT / 'plots'
SAMPLES_DIR = RESULTS_ROOT / 'sample_outputs'; REPORT_DIR = RESULTS_ROOT / 'reports'
for folder in [CHECKPOINT_ROOT, METRICS_DIR, PLOTS_DIR, SAMPLES_DIR, REPORT_DIR]: folder.mkdir(parents=True, exist_ok=True)
RESULTS_JSON = METRICS_DIR / 'benchmark_results.json'
records = json.loads(RESULTS_JSON.read_text(encoding='utf-8')) if RESULTS_JSON.exists() else []
for item in records:
    if item.get('status') == 'completed':
        item.pop('error', None); item.pop('traceback', None); item.pop('failure_class', None)
try: dataset_sha = HfApi().dataset_info(data_cfg['dataset_id'], revision=data_cfg['revision']).sha
except Exception as exc: dataset_sha = None; print('Dataset commit lookup unavailable:', exc)
try: model_sha = HfApi().model_info(selected_model, revision=model_cfg['revision']).sha
except Exception as exc: model_sha = model_cfg['revision']; print('Model commit lookup unavailable:', exc)
run_fingerprint = {'model': selected_model, 'model_revision': model_sha, 'dataset': data_cfg, 'dataset_revision': dataset_sha, 'training': train_cfg, 'mode': RUN_MODE}
run_id = hashlib.sha256(json.dumps(run_fingerprint, sort_keys=True).encode()).hexdigest()[:12]

splits, split_metadata = load_splits(
    data_cfg['dataset_id'], revision=dataset_sha or data_cfg['revision'], max_examples=data_cfg['max_examples'],
    seed=data_cfg['seed'], validation_fraction=data_cfg['validation_fraction'], test_fraction=data_cfg['test_fraction'])
print('Shared split sizes:', split_metadata['split_sizes'])
tokenizer = load_tokenizer(selected_model, model_sha)
max_length = min(data_cfg['max_length'], 512)
def tokenize_split(dataset):
    return dataset.map(lambda row: tokenize_example(row, tokenizer, max_length), remove_columns=dataset.column_names)
tokenized = {name: tokenize_split(split) for name, split in splits.items()}
if RUN_MODE == 'smoke':
    tokenized['train'] = tokenized['train'].select(range(min(len(tokenized['train']), train_cfg['smoke_examples'])))
print('Training rows in this run:', len(tokenized['train']))

common = {
    'run_id': run_id, 'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'run_mode': RUN_MODE,
    'selected_model_id': selected_model, 'model_revision': model_sha, 'selection_reason': hardware['selection_reason'],
    'hardware': hardware, 'full_finetuning_preflight': preflight_history, 'dataset_id': data_cfg['dataset_id'], 'dataset_revision': dataset_sha,
    'dataset_metadata': split_metadata, 'seed': train_cfg['seed'], 'max_length': max_length,
    'python_version': platform.python_version(),
    'package_versions': {},
    'training_config': train_cfg,
    'generation_config': {'do_sample': False, 'max_new_tokens': train_cfg['eval_max_new_tokens']},
}
versions = {}
for package in ['torch','transformers','datasets','peft','accelerate','bitsandbytes','rouge-score']:
    try: versions[package] = version(package)
    except PackageNotFoundError: versions[package] = None
common['package_versions'] = versions

def persist():
    save_results(records, METRICS_DIR)
    current_records = [item for item in records if item.get('run_id') == run_id]
    generate_plots(current_records, PLOTS_DIR)
    (REPORT_DIR / 'benchmark_report.md').write_text(build_report(current_records, PLOTS_DIR), encoding='utf-8')

def append_status(experiment, status, **values):
    row = next((item for item in reversed(records) if item.get('run_id') == run_id and item.get('experiment') == experiment), None)
    if row is None:
        row = {'experiment': experiment, **common}; records.append(row)
    if status == 'running':
        row.pop('error', None); row.pop('traceback', None); row.pop('failure_class', None)
    row.update({'status': status, **values}); persist(); return row

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


{
  "available": true,
  "name": "Tesla T4",
  "total_memory_gib": 14.56,
  "free_memory_gib": 14.46,
  "selection_reason": "primary model selected",
  "primary_model": "Qwen/Qwen2.5-0.5B-Instruct",
  "fallback_model": "HuggingFaceTB/SmolLM2-135M-Instruct",
  "full_finetuning_preflight": {
    "Qwen/Qwen2.5-0.5B-Instruct": {
      "fits": true,
      "reason": "scratch full-parameter optimizer step completed",
      "peak_gpu_memory_gib": 4.657374382019043,
      "probe_seconds": 40.872686464,
      "sequence_length": 512
    }
  }
}
Selected model for every experiment: Qwen/Qwen2.5-0.5B-Instruct
Shared split sizes: {'train': 960, 'validation': 120, 'test': 120}


Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Training rows in this run: 960


## Baseline

In [4]:
baseline_model = None
baseline_done = any(item.get('run_id') == run_id and item.get('experiment') == 'baseline' and item.get('status') == 'completed' for item in records)
if baseline_done:
    print('Baseline already completed for this config; reusing Drive results.')
else:
  try:
    row = append_status('baseline', 'running')
    baseline_model = load_model(selected_model, model_sha, quantized=False)
    meta = model_metadata(baseline_model, selected_model, tokenizer)
    quality, samples = evaluate_generation(baseline_model, tokenizer, splits['test'], train_cfg['eval_max_new_tokens'], max_length)
    quality['test_loss'] = evaluate_causal_loss(baseline_model, tokenizer, tokenized['test'])
    row.update({'status':'completed', 'error':None, 'traceback':None, 'failure_class':None, **meta, **quality, 'trainable_parameters':0, 'trainable_percent':0.0})
    (SAMPLES_DIR / f'{run_id}_baseline.json').write_text(json.dumps(samples, indent=2), encoding='utf-8')
    general = generate_generalization_samples(baseline_model, tokenizer, train_cfg['eval_max_new_tokens'], max_length)
    (SAMPLES_DIR / f'{run_id}_baseline_generalization.json').write_text(json.dumps(general, indent=2), encoding='utf-8')
    persist()
  except Exception as exc:
    append_status('baseline', 'failed', error=f'{type(exc).__name__}: {exc}', traceback=traceback.format_exc(limit=4))
    raise
  finally:
    if baseline_model is not None: del baseline_model
    gc.collect(); torch.cuda.empty_cache()

Baseline already completed for this config; reusing Drive results.


## Fine-tuning experiments

Each run writes a `running` row before model setup and updates it with measured results or an explicit failure. A full-fine-tuning OOM is recorded.

In [5]:
def run_finetune(experiment, method, rank=None):
    prior = next((item for item in reversed(records) if item.get('run_id') == run_id and item.get('experiment') == experiment), None)
    if prior and prior.get('status') == 'completed':
        print(f'{experiment} already completed for this config; skipping.')
        return prior
    row = append_status(experiment, 'running', method=method, rank=rank)
    model = None
    try:
        quantized = method == 'qlora'
        model = load_model(selected_model, model_sha, quantized=quantized)
        if method == 'lora':
            model = add_lora(model, rank=rank or 8, alpha=train_cfg['lora']['alpha'], dropout=train_cfg['lora']['dropout'], target_modules=train_cfg['lora']['target_modules'])
        elif method == 'qlora':
            model = add_qlora(model, rank=rank or 8, alpha=train_cfg['lora']['alpha'], dropout=train_cfg['lora']['dropout'], target_modules=train_cfg['lora']['target_modules'])
        elif method == 'full':
            model = prepare_full_finetuning(model)
        else: raise ValueError(f'Unknown method: {method}')
        counts = count_trainable_parameters(model)
        model_info = model_metadata(model, selected_model, tokenizer)
        output_dir = CHECKPOINT_ROOT / run_id / experiment
        measured = train_experiment(method, model, tokenizer, tokenized['train'], tokenized['validation'], output_dir, train_cfg, tokenized['test'])
        quality, samples = evaluate_generation(model, tokenizer, splits['test'], train_cfg['eval_max_new_tokens'], max_length)
        general = generate_generalization_samples(model, tokenizer, train_cfg['eval_max_new_tokens'], max_length)
        row.update({'status':'completed', 'error':None, 'traceback':None, 'failure_class':None, **model_info, **measured, **quality, 'rank':rank, 'checkpoint_dir':str(output_dir)})
        (SAMPLES_DIR / f'{run_id}_{experiment}.json').write_text(json.dumps(samples, indent=2), encoding='utf-8')
        (SAMPLES_DIR / f'{run_id}_{experiment}_generalization.json').write_text(json.dumps(general, indent=2), encoding='utf-8')
    except torch.cuda.OutOfMemoryError as exc:
        row.update({'status':'failed', 'error':f'CUDA out of memory: {exc}', 'failure_class':'oom'})
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as exc:
        row.update({'status':'failed', 'error':f'{type(exc).__name__}: {exc}', 'traceback':traceback.format_exc(limit=5)})
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    finally:
        if model is not None: del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        persist()
    return row

# Primary comparison uses rank 8 for both adapter methods.
run_finetune('lora', 'lora', rank=8)
run_finetune('qlora', 'qlora', rank=8)
run_finetune('full_finetuning', 'full')
# Rank 8 above is reused in the sweep; ranks 4, 16, and 32 complete the requested comparison.
for rank in [4, 16, 32]: run_finetune(f'lora_rank_{rank}', 'lora', rank=rank)

lora already completed for this config; skipping.
qlora already completed for this config; skipping.


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Epoch,Training Loss,Validation Loss


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:165: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,1.764800,1.761110


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,1.762600,1.756712


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,1.762700,1.754653


## Artifacts and generated report

Regenerates plots and the report from recorded rows, copies compact artifacts into the checked-out repository, and lists statuses.

In [6]:
current_records = [item for item in records if item.get('run_id') == run_id]
plots = generate_plots(current_records, PLOTS_DIR)
persist()
# Copy compact artifacts into the repository checkout for review/download.
local_metrics = pathlib.Path('results/metrics'); local_plots = pathlib.Path('results/plots')
local_samples = pathlib.Path('results/sample_outputs'); local_reports = pathlib.Path('results/reports')
for folder in [local_metrics, local_plots, local_samples, local_reports]: folder.mkdir(parents=True, exist_ok=True)
for source in [RESULTS_JSON, METRICS_DIR / 'benchmark_results.csv', REPORT_DIR / 'benchmark_report.md']:
    if source.exists():
        target = {'metrics':local_metrics, 'reports':local_reports}[source.parent.name] / source.name
        import shutil; shutil.copy2(source, target)
for source_dir, target_dir in [(PLOTS_DIR,local_plots),(SAMPLES_DIR,local_samples)]:
    for source in source_dir.iterdir():
        if source.is_file():
            import shutil; shutil.copy2(source, target_dir / source.name)
print('Statuses in this session:')
for item in current_records:
    detail = item.get('error','') if item.get('status') != 'completed' else ''
    print(item.get('experiment'), item.get('status'), detail)
print('Generated plots:', plots)
print('Durable results:', RESULTS_ROOT)

Statuses in this session:
baseline completed 
lora completed 
qlora completed 
full_finetuning completed 
lora_rank_4 completed 
lora_rank_16 completed 
lora_rank_32 completed 
Generated plots: ['/content/drive/MyDrive/llm-finetuning-benchmark/results/plots/quality_comparison.png', '/content/drive/MyDrive/llm-finetuning-benchmark/results/plots/training_time_comparison.png', '/content/drive/MyDrive/llm-finetuning-benchmark/results/plots/gpu_memory_comparison.png', '/content/drive/MyDrive/llm-finetuning-benchmark/results/plots/artifact_size_comparison.png', '/content/drive/MyDrive/llm-finetuning-benchmark/results/plots/trainable_parameter_percentage.png', '/content/drive/MyDrive/llm-finetuning-benchmark/results/plots/lora_rank_performance.png']
Durable results: /content/drive/MyDrive/llm-finetuning-benchmark/results
